# 00 — Getting Started with SDL 3.0 Case Notebooks

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TobiasEdman/space-data-lab/main?labpath=notebooks%2Fsdl3-cases%2F070_CASE_010-Getting-Started.ipynb)

**Purpose:** Template and onboarding notebook for the SDL 3.0 use-case library. Every case notebook follows this structure.

**Partners:** RISE · AI Sweden · Luleå tekniska universitet · Rymdstyrelsen

**Data sources:** ESA Copernicus Sentinel-2 L2A via [Digital Earth Sweden](https://digitalearth.se), CDSE fallback, or synthetic data for offline use.

**License:** CC0 1.0 Universal (notebook content). ImintEngine analyzers retain original licenses — see [THIRD_PARTY_LICENSES.md](https://github.com/TobiasEdman/imintengine/blob/main/THIRD_PARTY_LICENSES.md).

## 1. Method

This notebook demonstrates the **standard SDL 3.0 case-notebook structure** by running a minimal ImintEngine analysis over a small AOI.

**Analyzers used:** `spectral` (NDVI, NDWI).

**Expected output:** RGB composite, NDVI map, NDWI map, and metadata for a 1° × 1° AOI.

**Execution flow:**

```
AOI + date  →  LocalExecutor  →  fetch Sentinel-2  →  run_job()  →  visualize
                    │
                    └─ fallback to synthetic data if DES/CDSE unavailable
```

Subsequent case notebooks (01–06) use the same skeleton with different analyzers and AOIs.

## 2. Setup

Import dependencies and define the area of interest.

In [ ]:
# Verified API helper — wraps IMINTResult access
def get_outputs(result, name):
    """Return outputs dict for first successful analyzer matching `name`, else None."""
    for ar in result.analyzer_results:
        if ar.analyzer == name and ar.success:
            return ar.outputs
    return None


In [ ]:
from executors.local import LocalExecutor
import numpy as np
import matplotlib.pyplot as plt
import folium

# Small AOI — Gotland (~1° × 1°)
AOI = {
    "west": 18.0,
    "south": 57.2,
    "east": 19.0,
    "north": 57.8,
}
DATE = "2024-06-15"

print(f"AOI: {AOI}")
print(f"Date: {DATE}")

## 3. Run analysis

`LocalExecutor` handles data fetching, cloud detection, and analyzer orchestration. If no DES account is configured, it falls back to synthetic data so the notebook still runs end-to-end.

In [ ]:
executor = LocalExecutor(
    output_dir="outputs/getting_started",
    config_path="config/analyzers.yaml",
)

result = executor.execute(
    date=DATE,
    coords=AOI,
)

print(f"Job ID: {result.job_id}")
print(f"Analyzers run: {list(result.analyses.keys())}")

## 4. Visualize

Interactive map + static plots side by side.

In [ ]:
# Interactive Leaflet map centered on AOI
center = [(AOI["south"] + AOI["north"]) / 2, (AOI["west"] + AOI["east"]) / 2]
m = folium.Map(location=center, zoom_start=9, tiles="OpenStreetMap")
folium.Rectangle(
    bounds=[[AOI["south"], AOI["west"]], [AOI["north"], AOI["east"]]],
    color="#ff7800",
    weight=2,
    fill=True,
    fill_opacity=0.1,
    popup=f"AOI · {DATE}",
).add_to(m)
m

In [ ]:
# Static plots of analysis outputs
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

if get_outputs(result, "spectral") is not None:
    spec = get_outputs(result, "spectral")
    axes[0].imshow(result.rgb)
    axes[0].set_title("RGB")
    axes[0].axis("off")

    axes[1].imshow(spec.data["ndvi"], cmap="RdYlGn", vmin=-1, vmax=1)
    axes[1].set_title("NDVI")
    axes[1].axis("off")

    axes[2].imshow(spec.data["ndwi"], cmap="Blues", vmin=-1, vmax=1)
    axes[2].set_title("NDWI")
    axes[2].axis("off")

plt.tight_layout()
plt.show()

## 5. Interpretation & next steps

What this template demonstrated:

- **End-to-end flow:** AOI → data fetch → analyzer run → visualization in < 5 minutes
- **Reproducibility:** Every cell is deterministic given the same AOI/date + data source
- **Fallback strategy:** Synthetic data keeps the notebook runnable without DES credentials

### Use this as a template

To build a new case notebook:

1. Copy this notebook to `NN_case_name.ipynb`
2. Update the title, partners, and data sources in cell 1
3. Swap the AOI + date for your case
4. Enable case-specific analyzers in `config/analyzers.yaml`
5. Tailor the visualization to what your analysis outputs
6. Submit PR to [space-data-lab/notebooks/sdl3-cases/](https://github.com/TobiasEdman/space-data-lab/tree/main/notebooks/sdl3-cases)

### Links

- [ImintEngine source](https://github.com/TobiasEdman/imintengine)
- [SDL 3.0 application & deliverables](../../docs/sdl-phases.md)
- [Digital Earth Sweden platform](https://digitalearth.se)
- [DES community tutorials](https://github.com/DigitalEarthSweden/digital-earth-sweden-community)